
Build a Question Answer application over a Graph Database

In [1]:
NEO4J_URI="neo4j+s://30b8048e.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="C1OBpDqTp0KpPEjQVq-G53rXoAAawcBuU8sK-JqfG1I"

In [2]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [3]:
from langchain_community.graphs import Neo4jGraph
graph=Neo4jGraph(url=NEO4J_URI,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)
graph

In [14]:
moview_query = """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' AS row

WITH row
WHERE row.movieID IS NOT NULL AND row.movieID <> ""

MERGE (m:Movie {id: row.movieID})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

FOREACH (director IN split(row.director, '|') | 
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m))

FOREACH (actor IN split(row.actors, '|') | 
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))

FOREACH (genre IN split(row.genres, '|') | 
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

graph.query(moview_query)

[]

In [19]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("Groq_API_KEY")

In [25]:
from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000023B1ACD1610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023B1B070410>, model_name='llama-3.3-70b-versatile')

In [26]:
from langchain.chains import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(graph=graph,llm=llm,verbose=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x0000023B000BEDE0>, cypher_generation_chain=LLMChain(prompt=PromptTemplate(input_variables=['question', 'schema'], template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000023B1ACD1610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023B1B070410>, model_name='llama-3.3-70b-versatile')), qa_chain=LLMChain(prompt

In [28]:
response=chain.invoke({"query":"Who was the director of the movie casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'}) RETURN p.name
Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


{'query': 'Who was the director of the movie casino',
 'result': 'Martin Scorsese was the director of the movie Casino.'}

In [29]:
response=chain.invoke({"query":"Who was the actors of the movie casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: "Casino"})<-[:ACTED_IN]-(p:Person) RETURN p.name
Full Context:
[{'p.name': 'Robert De Niro'}, {'p.name': 'Joe Pesci'}, {'p.name': 'Sharon Stone'}, {'p.name': 'James Woods'}]

> Finished chain.


{'query': 'Who was the actors of the movie casino',
 'result': 'The actors in the movie Casino include Robert De Niro, Joe Pesci, Sharon Stone, and James Woods.'}